In [ ]:
#pip install codecarbon trl bitsandbytes peft

In [ ]:
#!pip install --user --force-reinstall transformers==5.1.0

In [ ]:
#%pip -q install -U torchao 
#%pip -q install -U transformers peft trl accelerate

In [ ]:
#pip install -U bitsandbytes

In [ ]:
#!pip -q install lm-eval langdetect

In [ ]:
#!pip install torchao==0.15.0 peft==0.17.0

In [ ]:
#!pip install --upgrade torchao
#pip install -U peft

In [ ]:
import os
from pathlib import Path
os.environ["HF_HUB_DISABLE_XET"] = "1"          # avoid CAS/Xet backend
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # more robust download path when available
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
#os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
os.environ["HF_HOME"] = str(Path.home() / ".cache" / "huggingface")
os.environ["PYTHONUTF8"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [ ]:
from codecarbon import EmissionsTracker, OfflineEmissionsTracker
import time, random, csv, torch, gc, json, subprocess, glob, shutil
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from torchao.quantization import quantize_, Int8DynamicActivationInt4WeightConfig
from torchao.quantization.qat import QATConfig
from peft import LoraConfig, TaskType, PromptTuningConfig, PromptTuningInit, prepare_model_for_kbit_training
from lm_eval import evaluator

In [ ]:
import torch
from peft.tuners.lora.dora import DoraLinearLayer
from peft.tuners.lora import dora as _dora_module

def _patched_get_weight_norm(self, weight, lora_weight, scaling, adapter_name=None):
    weight = _dora_module.transpose(weight, self.fan_in_fan_out)
    weight_fp32 = weight.to(torch.float32) + scaling * lora_weight.to(torch.float32)
    weight_norm = torch.linalg.norm(weight_fp32, dim=1)
    weight_norm = weight_norm.clamp(min=1e-6)          # never let it hit 0
    return weight_norm.to(weight.dtype)

DoraLinearLayer.get_weight_norm = _patched_get_weight_norm
print("DoRA get_weight_norm patched: fp32 norm + eps clamp")

In [ ]:
# import os
# LOCAL_RESULTS_DIR = "/content/ft_results"
# os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

# CSV_PATH = os.path.join(LOCAL_RESULTS_DIR, "results.csv")

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
CSV_PATH = NOTEBOOK_DIR / "results.csv"

In [ ]:
import os
os.environ["HF_TOKEN"] = ""

In [ ]:
from huggingface_hub import login
import os
login(token=os.environ["HF_TOKEN"])

In [ ]:
from huggingface_hub import snapshot_download, model_info
HF_TOKEN = os.environ["HF_TOKEN"]
LOCAL_MODEL_CACHE = {}

def assert_hf_access(repo_id: str):
    model_info(repo_id, token=HF_TOKEN)
    print(f"✅ HF access OK: {repo_id}")

def get_local_model_dir(repo_id: str) -> str:
    return snapshot_download(
        repo_id=repo_id,
        token=HF_TOKEN,
        resume_download=True,
    )

def get_local_model_dir_cached(repo_id: str) -> str:
    if repo_id not in LOCAL_MODEL_CACHE:
        LOCAL_MODEL_CACHE[repo_id] = get_local_model_dir(repo_id)
    return LOCAL_MODEL_CACHE[repo_id]

In [ ]:
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id="google/gemma-2-9b",
    filename="config.json",
)

In [ ]:
HEADERS = [
    "EXPERIMENT_ID",
    "ITERATION_NO",
    "DATASET_ID",
    "MODEL_ID",
    "FINE_TUNING_METHOD_ID",
    "FINE_TUNING_ENERGY_J",
    "FINE_TUNING_TIME_S",

    # lm-eval scores
    # "LM_EVAL_ARC_CHALLENGE",
    # "LM_EVAL_WINOGRANDE",
    "LM_EVAL_GPQA_DIAMOND_ZEROSHOT",
    "LM_EVAL_GPQA_DIAMOND_FEWSHOT",
    "LM_EVAL_MMLU",
    "LM_EVAL_TINYMMLU",
    "LM_EVAL_TRUTHFULQA_MC1",
    "LM_EVAL_MUSR",
    "LM_EVAL_TIME_S",
    "LM_EVAL_ENERGY_J",

    "ENTRY_DATA",
    "ENTRY_TIME_S",
]

def ensure_csv_header(csv_path):
    if not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(HEADERS)

ensure_csv_header(CSV_PATH)


# Dataset Preprocessing

In [ ]:
DATASET_REGISTRY = {
    "DS1_DOLLY_15K": {
        "hf_id": "databricks/databricks-dolly-15k",
        "split": "train",
        "formatter": "dolly15k_instruction_response",
    },
    "DS2_ALPACA_160K": {
        "hf_id": "tatsu-lab/alpaca",
        "split": "train",
        "formatter": "alpaca_instruction_input_output",
    },
    "DS3_OASST1_161K": {
        "hf_id": "OpenAssistant/oasst1",
        "split": "train",
        "formatter": "oasst1_best_reply_flatten",
    },
    "DS4_AYA_204K": {
        "hf_id": "CohereLabs/aya_dataset",
        "split": "train",
        "formatter": "aya_inputs_targets",
    },
}


In [ ]:
def format_dolly15k_dataset(example):
    instruction = example["instruction"].strip()
    context = (example.get("context") or "").strip()
    response = example["response"].strip()
    if context:
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Context:\n{context}\n\n"
            f"### Response:\n"
        )
    else:
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n"
        )
    return {"text": prompt + response}


def format_alpaca_dataset(example):
    instruction = example["instruction"].strip()
    input_ = example["input"].strip()
    output = example["output"].strip()

    if input_:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    return {"text": prompt + output}

def format_aya_dataset(example):
    input = (example.get("inputs") or "").strip()
    target = (example.get("targets") or "").strip()

    return {"text": f"### User:\n{input}\n\n### Assistant:\n{target}".strip()}

def format_oasst1_dataset(
    ds,
    only_lang=None, # "en" or None for all langs
    require_reviewed=True,
    drop_deleted=True,
):
    """
    Convert OASST1 into Instruction-Response samples by:
      - For each prompter message, choose the best assistant child (rank=0 best)
      - Output one text field: prompt + chosen_answer
    """

    if only_lang is not None:
        ds = ds.filter(lambda x: x["lang"] == only_lang)

    if require_reviewed and "review_result" in ds.column_names:
        ds = ds.filter(lambda x: x["review_result"] is True)

    if drop_deleted and "deleted" in ds.column_names:
        ds = ds.filter(lambda x: x["deleted"] is False)

    id2 = {}
    parent2children = {}

    for row in ds:
        mid = row["message_id"]
        pid = row["parent_id"]
        id2[mid] = row
        if pid is not None:
            parent2children.setdefault(pid, []).append(mid)

    samples = {"text": []}

    for parent_id, child_ids in parent2children.items():
        parent = id2.get(parent_id)
        if parent is None:
            continue
        if parent["role"] != "prompter":
            continue

        assistant_children = [id2[cid] for cid in child_ids if id2.get(cid, {}).get("role") == "assistant"]
        if not assistant_children:
            continue

        best = min(assistant_children, key=lambda r: r["rank"] if isinstance(r.get("rank"), (int, float)) else 10**9)

        prompt = parent["text"].strip()
        answer = best["text"].strip()

        samples["text"].append(
            f"### Instruction:\n{prompt}\n\n### Response:\n{answer}"
        )

    return Dataset.from_dict(samples)

In [ ]:
ROW_LEVEL_FORMATTER_MAP = {
    "dolly15k_instruction_response": format_dolly15k_dataset,
    "alpaca_instruction_input_output": format_alpaca_dataset,
    "aya_inputs_targets": format_aya_dataset,
}


DATASET_LEVEL_FORMATTER_MAP = {
    "oasst1_best_reply_flatten": format_oasst1_dataset,
}

DATASET_CACHE = {}

def preprocess_all_datasets(dataset_ids=None, only_lang_oasst=None):
    """
    Build processed 'text'-only datasets once and keep them in memory.
    """
    if dataset_ids is None:
        dataset_ids = list(DATASET_REGISTRY.keys())

    for ds_id in dataset_ids:
        cfg = DATASET_REGISTRY[ds_id]
        ds = load_dataset(cfg["hf_id"], split=cfg["split"])
        formatter_name = cfg["formatter"]

        if formatter_name in DATASET_LEVEL_FORMATTER_MAP:
            formatter_fn = DATASET_LEVEL_FORMATTER_MAP[formatter_name]
            processed = formatter_fn(ds, only_lang=only_lang_oasst)
        elif formatter_name in ROW_LEVEL_FORMATTER_MAP:
            formatter_fn = ROW_LEVEL_FORMATTER_MAP[formatter_name]
            processed = ds.map(
                formatter_fn,
                remove_columns=ds.column_names,
            )
        else:
            raise ValueError(f"Unknown formatter: {formatter_name}")

        assert "text" in processed.column_names, f"{ds_id} preprocessing failed: no 'text' column."

        # 80/20 split
        split_ds = processed.train_test_split(test_size=0.2, seed=42)

        train_ds = split_ds["train"]
        test_ds = split_ds["test"]

        DATASET_CACHE[ds_id] = {
            "train": train_ds,
            "test": test_ds,
        }

    return DATASET_CACHE

In [ ]:
def get_preprocessed_dataset(dataset_id, only_lang_oasst=None):
    # Return from cache if already prepared
    if dataset_id in DATASET_CACHE:
        return DATASET_CACHE[dataset_id]["train"]

    # Otherwise load + preprocess only the required dataset
    cfg = DATASET_REGISTRY[dataset_id]
    ds = load_dataset(cfg["hf_id"], split=cfg["split"])
    formatter_name = cfg["formatter"]

    if formatter_name in DATASET_LEVEL_FORMATTER_MAP:
        processed = DATASET_LEVEL_FORMATTER_MAP[formatter_name](ds, only_lang=only_lang_oasst)
    elif formatter_name in ROW_LEVEL_FORMATTER_MAP:
        processed = ds.map(
            ROW_LEVEL_FORMATTER_MAP[formatter_name],
            remove_columns=ds.column_names,
        )
    else:
        raise ValueError(f"Unknown formatter: {formatter_name}")

    split_ds = processed.train_test_split(test_size=0.2, seed=42)

    DATASET_CACHE[dataset_id] = {
        "train": split_ds["train"],
        "test": split_ds["test"],
    }

    return split_ds["train"]

# Model Loading

In [ ]:
def load_model_for_method(model_hf_id: str, method_id: str, use_bf16: bool):
    quant = METHODS[method_id]["quant"]
    dtype = torch.bfloat16 if use_bf16 else torch.float16
    assert_hf_access(model_hf_id)                 
    local_dir = get_local_model_dir_cached(model_hf_id)

    if quant == "torchao_qat":
        return AutoModelForCausalLM.from_pretrained(
            local_dir,
            torch_dtype=dtype,
            device_map="cuda:0",   # force on single GPU
            low_cpu_mem_usage=True,
        )

    if quant == "4bit":
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
        )
        return AutoModelForCausalLM.from_pretrained(
            local_dir,
            quantization_config=bnb,
            device_map="auto",
            low_cpu_mem_usage=True,
            torch_dtype=dtype,
        )

    if quant == "8bit":
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(
            local_dir,
            quantization_config=bnb,
            device_map="auto",
            offload_folder="./offload",
            offload_buffers=True,
            low_cpu_mem_usage=True,
        )

    return AutoModelForCausalLM.from_pretrained(
        local_dir,
        torch_dtype=dtype,
        device_map="auto",
        offload_folder="./offload",
        offload_buffers=True,
        low_cpu_mem_usage=True,
    )

In [ ]:
def guess_lora_targets(model):
    names = set()
    for n, m in model.named_modules():
        if m.__class__.__name__ in ("Linear", "Linear4bit", "Linear8bitLt"):
            names.add(n.split(".")[-1])

    # ordered from most expressive → safest
    candidates = [
        ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],  # LLaMA / Mistral / Gemma
        ["q_proj","k_proj","v_proj","o_proj"],                                    # attention only
        ["q_proj","v_proj"],                                                      # minimal safe
        ["fc1","fc2"],                                                            # Phi
    ]

    for c in candidates:
        if all(x in names for x in c):
            return c

    # final fallback: any q/v-like projections
    fallback = [n for n in names if "proj" in n or "fc" in n]
    if fallback:
        return sorted(fallback)

    raise RuntimeError(
        f"Could not infer LoRA targets. Found linear layers: {sorted(names)[:20]}"
    )

In [ ]:
def build_peft_config(model, method_id: str):
    peft_kind = METHODS[method_id]["peft"]
    if peft_kind is None:
        return None

    if peft_kind in ("lora", "dora"):
        if method_id == "QAT_LORA":
            targets = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
        else:
            targets = guess_lora_targets(model)

        return LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=16,
            lora_alpha=16,
            lora_dropout=0.00 if peft_kind == "dora" else 0.05,
            bias="none",
            target_modules=targets,
            use_dora=(peft_kind == "dora"),
        )

    if peft_kind == "prompt":
        return PromptTuningConfig(
            task_type=TaskType.CAUSAL_LM,
            num_virtual_tokens=10,
            prompt_tuning_init=PromptTuningInit.TEXT,
            prompt_tuning_init_text="You are a helpful assistant.",
            tokenizer_name_or_path=model.config._name_or_path,
        )

    raise ValueError(f"Unknown peft kind: {peft_kind}")

In [ ]:
def build_sft_config(run_name: str, method_id: str):
    # Default hyperparameters
    num_train_epochs = 3
    learning_rate = 2e-4
    gradient_accumulation_steps = 2
    lr_scheduler_type = "linear"
    warmup_steps = 100

    max_length = 512
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    if method_id == "SFT":
        num_train_epochs = 1  # or 2
        learning_rate = 2e-5  # typical full-finetune LR

    # Special hyperparameters for prompt tuning
    if method_id == "PROMPT_TUNING":
        num_train_epochs = 3         # more epochs
        learning_rate = 1e-2        # higher learning rate
        max_length = 256

    return SFTConfig(
        output_dir=f"outputs/{run_name}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        logging_steps=5,
        save_strategy="no",
        report_to=[],
        max_length=max_length,
        packing=False,
        dataset_text_field="text",
        gradient_checkpointing=(method_id != "DORA"),   # was: True
        lr_scheduler_type=lr_scheduler_type,
        warmup_steps=warmup_steps,
        fp16=not bf16,
        bf16=True,
        optim="paged_adamw_8bit",
    )

# Evaluation Harness

In [ ]:
import subprocess, sys, json, tempfile

def run_lm_eval(base_model_id, adapter_dir, method_id, tasks, device="cuda:0", limit=None):
    base_model_id = base_model_id.replace("\\", "/")
    adapter_dir   = adapter_dir.replace("\\", "/")

    cfg = {
        "base_model_id": base_model_id, "adapter_dir": adapter_dir,
        "method_id": method_id, "tasks": tasks, "device": device, "limit": limit,
    }

    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f_in:
        json.dump(cfg, f_in)
        cfg_path = f_in.name
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f_out:
        result_path = f_out.name

    print(f"  → Launching eval subprocess...")
    t0 = time.time()

    proc = subprocess.run(
        [sys.executable, "lm_eval_worker1.py", cfg_path, result_path],
        capture_output=True,   # was False -- this is what caused the old crash
        text=True, encoding="utf-8", errors="replace",
    )

    eval_time_s = time.time() - t0

    if proc.returncode not in (0, 2):   # 2 = worker's own "partial failure" code, still has usable output
        print(f"❌ lm-eval subprocess failed (returncode={proc.returncode})")
        print("─── STDERR (last 3000 chars) ───")
        print((proc.stderr or "")[-3000:])
        return {}, eval_time_s

    try:
        with open(result_path, "r") as f:
            payload = json.load(f)
        results_by_task = payload.get("results", {})
        errors_by_task  = payload.get("errors", {})
    except Exception as e:
        print(f"❌ Could not read eval results: {e}")
        return {}, eval_time_s

    if errors_by_task:
        print(f"⚠ task(s) failed inside worker: {list(errors_by_task.keys())}")
        for tname, err in errors_by_task.items():
            print(f"  {tname}: {err.get('error')}")

    for p in (cfg_path, result_path):
        try:
            os.unlink(p)
        except OSError:
            pass

    return results_by_task, eval_time_s

In [ ]:
LM_EVAL_TASKS = [
    # {"name": "arc_challenge", "fewshot": None},
    # {"name": "winogrande", "fewshot": None},
    {"name": "gpqa_diamond_zeroshot", "fewshot": 0},
    {"name": "gpqa_diamond_n_shot", "fewshot": 5},
    {"name": "mmlu", "fewshot": None},
    {"name": "tinyMMLU", "fewshot": None},
    {"name": "truthfulqa_mc1", "fewshot": None},
    {"name": "leaderboard_musr", "fewshot": None},
]

TASK_PRIMARY_METRIC = {
    # "arc_challenge": "acc_norm,none",
    # "winogrande": "acc,none",
    "gpqa_diamond_zeroshot": "acc,none",
    "gpqa_diamond_n_shot": "acc,none",
    "mmlu": "acc,none",
    "tinyMMLU": "acc,none",
    "truthfulqa_mc1": "acc,none",
    "leaderboard_musr": "acc_norm,none",
}

#def extract_task_score(results_json: dict, task_name: str):
#    res = results_json.get("results", {}).get(task_name, {})
#    if not isinstance(res, dict):
#        return None
#    metric = TASK_PRIMARY_METRIC.get(task_name)
#    if not metric:
#        return None
#    v = res.get(metric)
#    return v if isinstance(v, (int, float)) else None

def extract_task_score(results_json, task_name):

    if results_json is None:
        return None

    if not isinstance(results_json, dict):
        return None

    results = results_json.get("results")

    if not isinstance(results, dict):
        return None

    task = results.get(task_name)

    if not isinstance(task, dict):
        return None

    metric = TASK_PRIMARY_METRIC.get(task_name)

    if metric is None:
        return None

    return task.get(metric)


# Experimentation

In [ ]:
METHODS = {
    "SFT":           {"quant": None,    "peft": None},
    "LORA":          {"quant": None,    "peft": "lora"},
    "DORA":          {"quant": None,    "peft": "dora"},
    "QLORA":         {"quant": "4bit",  "peft": "lora"},
    "QAT_LORA":      {"quant": "torchao_qat", "peft": "lora"},
    "PROMPT_TUNING": {"quant": "4bit",    "peft": "prompt"},
}


In [ ]:
def energy_stats(energy_consumption_kwh, elapsed_s):
    energy_consumption_joules = energy_consumption_kwh * 1000 * 3600
    #duration = energy_tracker._last_measured_time - energy_tracker._start_time
    return energy_consumption_joules, elapsed_s

In [ ]:
import ctypes

def run_exp(exp_details):
    gc.collect()
    torch.cuda.empty_cache()
    # also force-release any cached HF model weights

    run_name = (
        f"exp{exp_details['experiment_id']}"
        f"_it{exp_details['iteration_no']}"
        f"_{exp_details['model_id']}"
        f"_{exp_details['dataset_id']}"
        f"_{exp_details['fine_tuning_method_id']}"
    )

    # ---------- ENTRY ----------
    exp_start = time.time()
    entry_start = time.time()
    train_ds = get_preprocessed_dataset(exp_details["dataset_id"])

    method_id = exp_details["fine_tuning_method_id"]

    sft_args = build_sft_config(run_name, method_id)
    local_dir = exp_details.get("local_model_dir") or get_local_model_dir_cached(exp_details["model_hf_id"])
    trainer = None

    model = load_model_for_method(exp_details["model_hf_id"], method_id, use_bf16=sft_args.bf16)
    model.config.use_cache = False

    if METHODS[method_id]["quant"] == "torchao_qat":
        base_config = Int8DynamicActivationInt4WeightConfig(group_size=32)
        quantize_(model, QATConfig(base_config, step="prepare"))

    if METHODS[method_id]["quant"] in {"4bit", "8bit"}:
        model = prepare_model_for_kbit_training(model)

    peft_config = build_peft_config(model, method_id)

    entry_data = f"rows={len(train_ds)}"
    entry_time = time.time() - entry_start

    # ---------- MEASURED TRAINING ----------
    tracker = OfflineEmissionsTracker(
        country_iso_code="ITA",
        project_name="ft_experiments",
        experiment_id=run_name,
        save_to_file=False,
    )

    tracker.start()
    train_wall_start = time.time()
    try:
        tokenizer = AutoTokenizer.from_pretrained(local_dir, use_fast=True)

        # --- FIX: pad + eos consistency (removes HF warning + avoids training issues)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        tokenizer.padding_side = "right"

        # --- IMPORTANT: align model config with tokenizer
        model.config.pad_token_id = tokenizer.pad_token_id
        model.config.eos_token_id = tokenizer.eos_token_id

        trainer = SFTTrainer(
            model=model,
            args=sft_args,
            train_dataset=train_ds,
            formatting_func=lambda x: x["text"],
            peft_config=peft_config,
            processing_class=tokenizer,
        )
        
        trainable = [n for n, p in trainer.model.named_parameters() if p.requires_grad]
        print(f"{len(trainable)} trainable params")
        print(trainable[:5])

        # --- DEBUG: manual forward+backward check ---
        sample_texts = [train_ds[i]["text"] for i in range(2)]
        enc = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(trainer.model.device)
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100

        trainer.model.train()
        dbg_out = trainer.model(**enc, labels=labels)
        print("DEBUG manual forward loss:", dbg_out.loss.item())
        dbg_out.loss.backward()
        grad_norms = {n: p.grad.norm().item() for n, p in trainer.model.named_parameters() if p.requires_grad and p.grad is not None}
        print("DEBUG params with grad:", len(grad_norms), "out of", len(trainable))
        print("DEBUG sample grads:", list(grad_norms.items())[:5])
        trainer.model.zero_grad()  # clear so it doesn't interfere with the real training step
        # --- END DEBUG ---

        adapter_dir = f"adapters/{run_name}"
        os.makedirs(adapter_dir, exist_ok=True)
        trainer.train()

        if METHODS[method_id]["quant"] == "torchao_qat":
            # 1) Merge LoRA weights into the base model
            merged = trainer.model.merge_and_unload()  # PEFT API

            # 2) Convert fake-quant to real quantized model
            base_config = Int8DynamicActivationInt4WeightConfig(group_size=32)
            quantize_(merged, QATConfig(base_config, step="convert"))

            # 3) Save full quantized model (like SFT)
            merged.save_pretrained(adapter_dir, safe_serialization=False)
            tokenizer.save_pretrained(adapter_dir)
        else:
            trainer.model.save_pretrained(adapter_dir, safe_serialization=False)
            tokenizer.save_pretrained(adapter_dir)
        print(f"✅ Fine-tuning done: {(time.time() - exp_start) / 60:.1f} min since exp start")


   # finally:
        #kwh = tracker.stop()
        #if trainer is not None:
        #    del trainer
        #if model is not None:
       #     del model
       # gc.collect()
       # torch.cuda.empty_cache()
       # time.sleep(5)
    finally:
        kwh = tracker.stop()
        train_wall_end = time.time()

        try:
            del trainer
        except:
            pass

        try:
            del model
        except:
            pass

        try:
            del tokenizer
        except:
            pass

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        time.sleep(10)

    ft_energy_j, ft_time_s = energy_stats(kwh, train_wall_end - train_wall_start)

    eval_tracker = OfflineEmissionsTracker(
        country_iso_code="ITA",
        project_name="ft_experiments",
        experiment_id=run_name + "_eval",
        save_to_file=False,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    time.sleep(10)

    eval_tracker.start()
    #eval_t0 = time.time()

    eval_results_by_task = {}
    eval_time_s = 0.0

    #print(f"▶ Running all lm-eval tasks in one subprocess...")
    #eval_results_by_task, eval_time_s = run_lm_eval(
    #    base_model_id=local_dir,
    #    adapter_dir=adapter_dir,
    #    method_id=method_id,
    #    tasks=LM_EVAL_TASKS,
    #    device="cuda:0",
    #    # limit=2
    #)

    print(f"▶ Running lm-eval tasks one by one...")

    eval_results_by_task = {}
    eval_time_s = 0.0

    for task in LM_EVAL_TASKS:
        print(f"▶ Running task: {task['name']}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        try:
            task_result, task_time = run_lm_eval(
                base_model_id=local_dir,
                adapter_dir=adapter_dir,
                method_id=method_id,
                tasks=[task],
                device="cuda:0",
                limit=None
            )

            print(f"{task['name']} returned:")
            print(task_result)

            eval_time_s += task_time

            if isinstance(task_result, dict):
                eval_results_by_task.update(task_result)
            else:
                print(f"⚠ {task['name']} returned {type(task_result)}")

        except Exception as e:
            print(f"⚠ {task['name']} failed: {e}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        time.sleep(5)



    print(f"✅ lm-eval done: {(time.time() - exp_start) / 60:.1f} min since exp start")

    eval_kwh = eval_tracker.stop()
    eval_energy_j = eval_kwh * 3_600_000

    # arc_challenge = extract_task_score(eval_results_by_task.get("arc_challenge", {}), "arc_challenge")
    # winogrande = extract_task_score(eval_results_by_task.get("winogrande", {}), "winogrande")
    gpqa_zero = extract_task_score(eval_results_by_task.get("gpqa_diamond_zeroshot", {}), "gpqa_diamond_zeroshot")
    gpqa_few = extract_task_score(eval_results_by_task.get("gpqa_diamond_n_shot", {}), "gpqa_diamond_n_shot")
    mmlu_score = extract_task_score(eval_results_by_task.get("mmlu", {}), "mmlu")
    tinymmlu_score = extract_task_score(eval_results_by_task.get("tinyMMLU", {}), "tinyMMLU")
    truthfulqa_mc1 = extract_task_score(eval_results_by_task.get("truthfulqa_mc1", {}), "truthfulqa_mc1")
    musr = extract_task_score(eval_results_by_task.get("leaderboard_musr", {}), "leaderboard_musr")


    row = [
        exp_details["experiment_id"],
        exp_details["iteration_no"],
        exp_details["dataset_id"],
        exp_details["model_id"],
        method_id,
        ft_energy_j,
        ft_time_s,
        # arc_challenge,
        # winogrande,
        gpqa_zero,
        gpqa_few,
        mmlu_score,
        tinymmlu_score,
        truthfulqa_mc1,
        musr,
        eval_time_s,
        eval_energy_j,
        entry_data,
        entry_time,
    ]
    with open(CSV_PATH, "a", newline="") as f:
        csv.writer(f).writerow(row)

    # cleanup disk: delete adapters + lm_eval outputs for this run
    # shutil.rmtree(adapter_dir, ignore_errors=True)


In [ ]:
CSV_PATH

In [ ]:
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

def warmup(duration_seconds=120):
    start = time.time()

    # CPU warmup
    while time.time() - start < duration_seconds / 2:
        fibonacci(40)

    # GPU warmup if CUDA is available
    if torch.cuda.is_available():
        device = torch.device("cuda")
        x = torch.randn(2048, 2048, device=device)
        for _ in range(50):
            x = x @ x
            torch.cuda.synchronize()

In [ ]:
DATASETS = [
    #{"id": "DS1_DOLLY_15K", "name": "Dolly 15k"},
    {"id": "DS2_ALPACA_160K", "name": "Alpaca 160k"},
    #{"id": "DS3_OASST1_161K", "name": "OASST1 161k"},
    #{"id": "DS4_AYA_204K", "name": "Aya 204k"},
]

FINE_TUNING_METHODS = [
    #{"id": "SFT", "name": "Supervised Fine-Tuning"},
    #{"id": "LORA", "name": "LoRA"},                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            {"id": "LORA", "name": "LoRA"},
    #{"id": "QLORA", "name": "QLoRA"},
    {"id": "DORA", "name": "DoRA"},
    #{"id": "PROMPT_TUNING", "name": "Prompt Tuning"},
]

MODELS = [
    #{"id": "LLAMA32_1B", "hf": "meta-llama/Llama-3.2-1B", "name": "Llama-3.2 1B"},
    {"id": "LLAMA32_3B", "hf": "meta-llama/Llama-3.2-3B", "name": "Llama-3.2 3B"},
    #{"id": "DS_R1_QWEN_1P5B", "hf": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "name": "DeepSeek-R1-Distill-Qwen-1.5B"},
    #{"id": "DS_R1_QWEN_7B", "hf": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "name": "DeepSeek-R1-Distill-Qwen-7B"},
    #{"id": "MISTRAL_7B", "hf": "mistralai/Mistral-7B-v0.1", "name": "Mistral 7B"},
    #{"id": "MISTRAL_7B_V02", "hf": "mistralai/Mistral-7B-Instruct-v0.2", "name": "Mistral 7B v0.2"},
    #{"id": "GEMMA_9B", "hf": "google/gemma-2-9b", "name": "Gemma 9B"},
    #{"id": "PHI3_MINI", "hf": "microsoft/Phi-3-mini-4k-instruct", "name": "Phi 3 mini"},
]
NUMBER_OF_EXPERIMENTAL_RUNS = 1
EXPERIMENT_ID = 0
EXPERIMENTS = []

for ds in DATASETS:
  for ftm in FINE_TUNING_METHODS:
    for model in MODELS:
      EXPERIMENT_ID += 1
      for iteration in range(NUMBER_OF_EXPERIMENTAL_RUNS) :
        EXPERIMENTS.append ({
          "experiment_id": EXPERIMENT_ID,
          "iteration_no": iteration,
          "dataset_id": ds["id"],
          "dataset_name": ds["name"],
          "fine_tuning_method_id": ftm["id"],
          "fine_tuning_method_name": ftm["name"],
          "model_id": model["id"],
          "model_name": model["name"],
          "model_hf_id": model["hf"],
        })


print(f"Total runs = {len(EXPERIMENTS)} "
      f"({len(DATASETS)} datasets × {len(FINE_TUNING_METHODS)} methods × "
      f"{len(MODELS)} models × {NUMBER_OF_EXPERIMENTAL_RUNS} repeats)")

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=memory.free,memory.used", 
                        "--format=csv"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# random.shuffle(EXPERIMENTS)
preprocess_all_datasets()

warmup()

import traceback

for exp in EXPERIMENTS:
    print(f"********** Experiment {exp['experiment_id']} *************")
    print(f"********** Model: {exp['model_id']}, Fine tuning: {exp['fine_tuning_method_id']}, Dataset: {exp['dataset_id']} *************")
    time.sleep(10)
    try:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             
        # 1) fail if gated/auth is wrong
        assert_hf_access(exp["model_hf_id"])

        # 2) prefetch + reuse local snapshot path
        exp["local_model_dir"] = get_local_model_dir_cached(exp["model_hf_id"])

        run_exp(exp)

    except Exception:
        traceback.print_exc()

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

In [ ]:
import os
print(os.listdir("adapters"))
print(os.path.exists("lm_eval_worker1.py"))   # this is what run_lm_eval() actually calls

In [ ]:
run_name      = "exp1_it0_LLAMA32_3B_DS1_DOLLY_15K_DORA"
experiment_id = 1
iteration_no  = 0
dataset_id    = "DS1_DOLLY_15K"
model_id      = "LLAMA32_3B"
model_hf_id   = "meta-llama/Llama-3.2-3B"
method_id     = "DORA"

In [ ]:
import inspect
print(inspect.getsource(energy_stats))